In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MHA(nn.Module):
    def __init__(self, hidden_dim, num_head, dropout_rate=0.1) -> None:
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_head = num_head
        self.head_dim = hidden_dim // num_head
        self.dropout_rate = dropout_rate

        self.q_proj = nn.Linear(self.hidden_dim,self.hidden_dim)
        self.k_proj = nn.Linear(self.hidden_dim,self.hidden_dim)
        self.v_proj = nn.Linear(self.hidden_dim,self.hidden_dim)

        self.dropout = nn.Dropout(self.dropout_rate)

        self.o_proj = nn.Linear(self.hidden_dim,self.hidden_dim)


    def forward(self,x,mask=None):
        # x (b,seq_len,h)
        b,s = x.shape[0],x.shape[1]
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        # (b,num_head,s,h)
        Q = Q.view(b,s,self.num_head,self.head_dim).transpose(1,2)
        K = K.view(b,s,self.num_head,self.head_dim).transpose(1,2)
        V = V.view(b,s,self.num_head,self.head_dim).transpose(1,2)
        
        # (b,num_head,s,s)
        atten = (Q @ K.transpose(-1,-2)/math.sqrt(self.head_dim))
        if mask is not None:
            atten = atten.masked_fill(mask==0,float("-inf"))
        
        atten = self.dropout(torch.softmax(atten,dim=-1))
        
        output = atten @ V
        output = output.transpose(1,2).contiguous().view(b, s, -1)
        output = self.o_proj(output)

        return output


x = torch.rand(3, 2, 128)
net = MHA(128, 8)
net(x).shape

torch.Size([3, 2, 128])

In [27]:
b, s, h = 2, 4, 8  # batch, seq_len, hidden_dim
x = torch.rand(b, s, h)

mask = torch.ones(b,s)
mask[:,3:] = 0

mask = mask.unsqueeze(1).unsqueeze(1)  # 变成 (b, 1, 1, s) -> broadcast 到 num_head, seq_len

mha = MHA(hidden_dim=h, num_head=2)
out = mha(x, mask)
print("输出形状:", out.shape)  # (b, s, h)

输出形状: torch.Size([2, 4, 8])
